### Interactive debugging tool

In [1]:
import ipywidgets as widgets
from IPython.display import display
from util.data_aquisition import PbPData
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
%matplotlib inline

#### Helper function

In [2]:
def extract_game_name(game):
    game_id = game['gamePk']
    home = game['gameData']['teams']['home']['name']
    away = game['gameData']['teams']['away']['name']
    dateTime = game['gameData']['datetime']['dateTime']
    return str(game_id)+ ': '+ home+ ' vs '+away + ' '+ dateTime+'.'

#### Code for the widget

Loading the nhl field image

In [3]:
img = mpimg.imread('figures/nhl_rink.png')

Defining the widgets we will include

In [4]:
season = widgets.Dropdown( #Dropdown menu to select the season
    options = ['2016-2017', '2017-2018', '2018-2019', '2019-2020', '2020-2021'],
    value = '2016-2017',
    description= 'Season: ',
    disabled=False,
    layout = widgets.Layout(min_height='100px')
)
play_off = widgets.ToggleButtons( #Buttons to choose between regular or play-off
    options=['regular', 'play-off'],
    description='Game type:',
    disabled=False,
    button_style='', 
    
)
load_data = widgets.Button(description = "Load data") #Button to confirm the choice of season and game type and load
#the data to the memory
output = widgets.Output() #Output widget to print information about whether or not data is loaded, or being loaded
#or downloaded
with output: #Initialization of the output widget: at first no data is loaded.
    print ('No data loaded. Please select season and game type and press the load data button.')
    
    
games = widgets.Dropdown(#Dropdown menu to select the game
    options =[],
    description ='Game: ',
    disabled= True,
    )
games_output = widgets.Output() #Output widget to print information about the selected game
events_slider = widgets.IntSlider(#Slider to select the event in the game you want to look up
    step = 1,
    disabled = True,
    description = 'Event: ',
)
event_output = widgets.Output()#Output widget to print the event info and coordinates plot if available


Defining callbacks

In [5]:
def load_callback(b): #Callback to be triggered whenever the button load_data is clicked
    config = {'saison': int(season.value[0:4]), 'play_off': play_off.value == 'play-off'}
    output.clear_output()
    global pbp_data
    with output:
        pbp_data = PbPData(config)
    
    output.clear_output()
    with output:
        print('Current loaded data: Season '+ season.value+' '+play_off.value+' games.')
    global games_data
    games_data = pbp_data.data[0]['data']
    games_data.sort(key = lambda g: g['gamePk'])
    games_titles = [(extract_game_name(game),i) for i,game in enumerate(games_data)]
    games.disabled = False
    games.options = games_titles
    
def on_game_change(change): #Callback to be triggered whenever a new game is chosen
    
    #Changes related to games_output
    games_output.clear_output()
    with games_output:
        game = games_data[change.new]
        game_id = game['gamePk']
        home = game['gameData']['teams']['home']['name']
        away = game['gameData']['teams']['away']['name']
        dateTime = game['gameData']['datetime']['dateTime']
        date = dateTime.split('T')[0]
        time = dateTime.split('T')[1][:-1]
        n_events = len(game['liveData']['plays']['allPlays'])
        print('GameID: '+str(game_id))
        print('Home: '+home)
        print('Away: '+away)
        print('Date: '+date)
        print('Time: '+time)
        print('Number of events: '+str(n_events)+'.')
        
    #Changes related to the slider events
    events_slider.disabled = False
    events_slider.min=1
    events_slider.value=1
    events_slider.max=n_events
    
    
def on_event_change(change): #Callback to be triggered whenever a new event is chosen
    game = games_data[games.value]
    event = game['liveData']['plays']['allPlays'][change.new-1]
    event_output.clear_output()
    coordinates = event['coordinates']
    has_coordinates = bool(coordinates)
    with event_output:
        if has_coordinates: #plot coordinates if they exist
            fig = plt.figure()
            ax = fig.add_subplot(111)
            ax.imshow(img, extent = [-100, 100, -42.5, 42.5])
            ax.scatter(x=[coordinates['x']],y=[coordinates['y']], c='black', marker='D')
            plt.show()
        print(json.dumps(event, indent=4)) #print json event info

Linking widgets to callback

In [6]:
load_data.on_click(load_callback)
games.observe(on_game_change, names= 'value')
events_slider.observe(on_event_change, names='value')

Creating a new widgets containing all the widgets above and displaying it

In [7]:
final_widget = widgets.VBox([widgets.HBox([season, play_off, load_data]), output, games, games_output, events_slider, event_output])
display(final_widget)